# Complete BaBar isobar CP closure: $B^\pm\to K^\pm\pi^\mp\pi^\pm$

This notebook implements the **complete nominal signal Dalitz model** of BaBar, Phys. Rev. D 78, 012004 (2008), arXiv:0803.4451. The paper uses
\[c_j=(x_j+\Delta x_j)+i(y_j+\Delta y_j),\qquad \bar c_j=(x_j-\Delta x_j)+i(y_j-\Delta y_j),\]
which is exactly `CPRealImag`. The central Cartesian values below are taken directly from Table I.

Particle ordering is fixed to `(1,2,3) = (K^\pm, pi^\pm, pi^\mp)`. Therefore the Dalitz plane used throughout this notebook is
\[s_{13}=m^2(K^\pm\pi^\mp),\qquad s_{23}=m^2(\pi^+\pi^-).\]

The nominal model contains a constant nonresonant term plus nine intermediate states: $K^*(892)^0$, the LASS $(K\pi)_0^{*0}$ S-wave, $K_2^*(1430)^0$, $\rho(770)^0$, $\omega(782)$, $f_0(980)$, $f_2(1270)$, the scalar $f_X(1300)$, and $\chi_{c0}$.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, CPRealImag, DecayChannel, DecayModel, LASS, Minimizer,
    NonResonant, Parameter, Resonance, enable_x64, weighted_resample,
)
from dalitzplotfitter.likelihood import SimultaneousNLL

enable_x64()

## 1. Table-I Cartesian coefficients

`fixed_cp=True` reproduces the BaBar choice $\Delta x=\Delta y=0$ for the $\omega(782)K$ and phase-space nonresonant terms. The $K^*(892)^0$ reference has fixed $x=1,y=0$ but its CP-odd Cartesian shifts are allowed to float, as in the paper.

In [ ]:
TABLE_I = {
    # name: (x, y, dx, dy, fixed_xy, fixed_cp)
    'Kstar892':    ( 1.000,  0.000, -0.017, -0.238, True,  False),
    'KpiS':        ( 1.718, -0.727, -0.154, -0.285, False, False),
    'rho770':      ( 0.683, -0.025, -0.160,  0.169, False, False),
    'f0_980':      (-0.220,  1.203, -0.109,  0.047, False, False),
    'chic0':       (-0.263,  0.180, -0.033, -0.007, False, False),
    'NR':          (-0.594,  0.068,  0.000,  0.000, False, True),
    'K2star1430':  (-0.301,  0.424,  0.032,  0.007, False, False),
    'omega782':    (-0.058,  0.100,  0.000,  0.000, False, True),
    'f2_1270':     (-0.193,  0.110, -0.089,  0.125, False, False),
    'fX1300':      (-0.290, -0.136,  0.024,  0.056, False, False),
}

def cp_coefficient(name):
    x, y, dx, dy, fixed_xy, fixed_cp = TABLE_I[name]
    def p(suffix, value, fixed=False, bound=3.0, step=0.02):
        return Parameter.coefficient(
            f'{name}.{suffix}', value, owner=name, fixed=fixed,
            bounds=(-bound, bound), step=step,
        )
    return CPRealImag(
        p('x', x, fixed_xy), p('y', y, fixed_xy),
        p('dx', dx, fixed_cp, 1.5, 0.01), p('dy', dy, fixed_cp, 1.5, 0.01),
    )

coeff = {name: cp_coefficient(name) for name in TABLE_I}

## 2. Complete nominal dynamical model

BaBar uses $r_{BW}=4\,\mathrm{GeV}^{-1}$, unit-normalized dynamical functions, RBW for ordinary resonances, LASS with $a=2.07\,\mathrm{GeV}^{-1}$, $r=3.32\,\mathrm{GeV}^{-1}$ and a 1.8 GeV cutoff for the $K\pi$ S-wave, and a Flatte form for $f_0(980)$ with $g_\pi=0.165$ GeV and $g_K=4.21g_\pi$. The fitted scalar $f_X$ uses $m=1.479$ GeV and $\Gamma=0.080$ GeV.

In [ ]:
channel_plus  = DecayChannel('B+', ('K+', 'pi+', 'pi-'))
channel_minus = DecayChannel('B-', ('K-', 'pi-', 'pi+'))

def build_model(channel, charge):
    c = lambda name: coeff[name].for_charge(charge)
    R = 4.0
    return DecayModel(channel, [
        Resonance('Kstar892',   (0,2), c('Kstar892'),   mass=0.8958, width=0.0474, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('KpiS',       (0,2), c('KpiS'),       lineshape=LASS(2.07,3.32,1.8), mass=1.425, width=0.270, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('rho770',     (1,2), c('rho770'),     mass=0.7753, width=0.1491, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f0_980',     (1,2), c('f0_980'),     lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=R, parent_radius=R),
        Resonance('chic0',      (1,2), c('chic0'),      mass=3.4147, width=0.0105, spin=0, resonance_radius=R, parent_radius=R),
        NonResonant(c('NR'), name='NR'),
        Resonance('K2star1430', (0,2), c('K2star1430'), mass=1.4324, width=0.109, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('omega782',   (1,2), c('omega782'),   mass=0.78265, width=0.00849, spin=1, resonance_radius=R, parent_radius=R),
        Resonance('f2_1270',    (1,2), c('f2_1270'),    mass=1.2755, width=0.1867, spin=2, resonance_radius=R, parent_radius=R),
        Resonance('fX1300',     (1,2), c('fX1300'),     mass=1.479, width=0.080, spin=0, resonance_radius=R, parent_radius=R),
    ], normalization_resolution=350)

model_plus = build_model(channel_plus, +1)
model_minus = build_model(channel_minus, -1)
truth = {p.name: float(p.value) for p in model_plus.parameters}
print('free parameters:', sum(not p.fixed for p in model_plus.parameters))

## 3. Generate independent charge toys and plot the correct Dalitz coordinates

In [ ]:
N_POOL, N_TOY = 500_000, 60_000
pool_plus = model_plus.generate_phase_space(N_POOL, seed=78012004)
pool_minus = model_minus.generate_phase_space(N_POOL, seed=78012005)
w_plus = pool_plus.weights * model_plus.intensity(pool_plus.as_dict(), truth)
w_minus = pool_minus.weights * model_minus.intensity(pool_minus.as_dict(), truth)
toy_plus = weighted_resample(jax.random.key(78012006), pool_plus, w_plus, N_TOY, replace=True)
toy_minus = weighted_resample(jax.random.key(78012007), pool_minus, w_minus, N_TOY, replace=True)

fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax, toy, title in [(axes[0],toy_plus,r'$B^+$'),(axes[1],toy_minus,r'$B^-$')]:
    h=ax.hist2d(np.asarray(toy.s13),np.asarray(toy.s23),bins=100)
    fig.colorbar(h[3],ax=ax,label='events')
    ax.set(xlabel=r'$s_{13}=m^2(K^\pm\pi^\mp)$ [GeV$^2$]', ylabel=r'$s_{23}=m^2(\pi^+\pi^-)$ [GeV$^2$]', title=title)
plt.show()

## 4. $s_{13}$ and $s_{23}$ projections and raw toy asymmetries

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12,8),constrained_layout=True)
for col,(attr,label) in enumerate([('s13',r'$s_{13}=m^2(K\pi)$'),('s23',r'$s_{23}=m^2(\pi\pi)$')]):
    vp=np.asarray(getattr(toy_plus,attr)); vm=np.asarray(getattr(toy_minus,attr))
    bins=np.linspace(min(vp.min(),vm.min()),max(vp.max(),vm.max()),100)
    hp,e=np.histogram(vp,bins=bins); hm,_=np.histogram(vm,bins=bins); x=0.5*(e[:-1]+e[1:])
    axes[0,col].step(x,hp,where='mid',label=r'$B^+$'); axes[0,col].step(x,hm,where='mid',label=r'$B^-$'); axes[0,col].legend(); axes[0,col].set_ylabel('events')
    axes[1,col].axhline(0,lw=.8); axes[1,col].step(x,(hm-hp)/np.maximum(hm+hp,1),where='mid'); axes[1,col].set(xlabel=label+' [GeV$^2$]',ylabel=r'$(N_- - N_+)/(N_- + N_+)$')
plt.show()

## 5. Cached simultaneous likelihood and one-start closure fit

Only coefficient parameters float. The ten dynamical columns and normalization matrices are cached once for each charge.

In [ ]:
cache_plus=model_plus.prepare_cache(toy_plus)
cache_minus=model_minus.prepare_cache(toy_minus)

def nll(cache):
    def f(values):
        intensity,norm=cache.evaluate(values); tiny=jnp.finfo(intensity.dtype).tiny
        return -jnp.sum(jnp.log(jnp.maximum(intensity,tiny)))+intensity.shape[0]*jnp.log(norm)
    return f

objective=SimultaneousNLL((nll(cache_plus),nll(cache_minus)))
parameters=model_plus.parameters
fitter=Minimizer(objective,parameters,tolerance=1e-5,verbose=1)
start=fitter.random_start(seed=20260830)
result=fitter.fit(start_values=start,simplex=False,ncall=50000)
print(result.fmin)

## 6. Truth/start/fit/pull and reconstructed component $A_{CP}$

In [ ]:
free=[p for p in parameters if not p.fixed]
fit={p.name:float(result.values[p.name]) for p in free}; err={p.name:float(result.errors[p.name]) for p in free}
print(f"{'parameter':18s} {'truth':>9s} {'start':>9s} {'fit':>9s} {'err':>9s} {'pull':>8s}")
for p in free:
    pull=(fit[p.name]-truth[p.name])/err[p.name]
    print(f"{p.name:18s} {truth[p.name]:9.4f} {start[p.name]:9.4f} {fit[p.name]:9.4f} {err[p.name]:9.4f} {pull:8.2f}")

def acp(c,values):
    cp=complex(c.for_charge(+1).value(values)); cm=complex(c.for_charge(-1).value(values))
    return (abs(cm)**2-abs(cp)**2)/(abs(cm)**2+abs(cp)**2)

print('\ncomponent ACP:')
for name,c in coeff.items():
    print(f'{name:12s} truth={acp(c,truth):+.4f}  fit={acp(c,fit):+.4f}')